# Final Project: Hotel Booking Cancellation Prediction

## Business Questions

1. What booking features are most related to cancellation?
2. Which customer or market segments cancel more often?
3. Which machine learning model gives the best prediction performance?

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 100)

# Data Exploration

In [ ]:
df = pd.read_csv('hotel_bookings.csv')
df

In [ ]:
df.info()

In [ ]:
df.describe().round(2)

In [ ]:
df.describe(include=['object'])

In [ ]:
df.duplicated().sum()

In [ ]:
df.isna().sum()

In [ ]:
df.isna().mean().round(4) * 100

# Data Cleaning

In [ ]:
df.drop_duplicates(inplace=True, ignore_index=True)
df = df.rename(columns={
    'Booking_ID': 'booking_id',
    'number of adults': 'number_of_adults',
    'number of children': 'number_of_children',
    'number of weekend nights': 'number_of_weekend_nights',
    'number of week nights': 'number_of_week_nights',
    'type of meal': 'type_of_meal',
    'car parking space': 'car_parking_space',
    'room type': 'room_type',
    'lead time': 'lead_time',
    'market segment type': 'market_segment_type',
    'repeated': 'repeated',
    'P-C': 'previous_cancellations',
    'P-not-C': 'previous_non_cancellations',
    'average price ': 'average_price',
    'special requests': 'special_requests',
    'date of reservation': 'reservation_date',
    'booking status': 'booking_status'
})
df['reservation_date'] = pd.to_datetime(df['reservation_date'], format='%m/%d/%Y', errors='coerce')
numeric_columns = ['number_of_adults', 'number_of_children', 'number_of_weekend_nights', 'number_of_week_nights', 'car_parking_space', 'lead_time', 'repeated', 'previous_cancellations', 'previous_non_cancellations', 'average_price', 'special_requests']
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors='coerce')
df['type_of_meal'] = df['type_of_meal'].fillna('Unknown')
df['room_type'] = df['room_type'].fillna('Unknown')
df['market_segment_type'] = df['market_segment_type'].fillna('Unknown')
df.dropna(subset=['reservation_date', 'booking_status'], inplace=True)
df.info()

# Feature Engineering

In [ ]:
df['reservation_month'] = df['reservation_date'].dt.month
df['reservation_day'] = df['reservation_date'].dt.day
df['reservation_weekday'] = df['reservation_date'].dt.dayofweek
df['total_guests'] = df['number_of_adults'] + df['number_of_children']
df['total_nights'] = df['number_of_weekend_nights'] + df['number_of_week_nights']
history_total = df['previous_cancellations'] + df['previous_non_cancellations']
df['cancellation_ratio'] = df['previous_cancellations'].div(history_total.where(history_total > 0, 1)).fillna(0.0)
df['arrival_is_weekend'] = df['reservation_weekday'].isin([4, 5]).astype(int)
df['lead_time_level'] = pd.cut(df['lead_time'], bins=[-1, 1, 7, 30, 90, df['lead_time'].max()], labels=['same_day', 'short', 'medium', 'planned', 'long'])
df[['total_guests', 'total_nights', 'cancellation_ratio', 'arrival_is_weekend', 'lead_time_level']].head()

# EDA

In [ ]:
df[['number_of_adults', 'number_of_children', 'lead_time', 'average_price', 'special_requests', 'total_guests', 'total_nights', 'cancellation_ratio']].describe().T

In [ ]:
df['booking_status'].value_counts(normalize=True).mul(100).round(2)

In [ ]:
sns.countplot(data=df, x='booking_status')
plt.title('Booking Status Distribution')
plt.xticks(rotation=15)
plt.show()

In [ ]:
sns.histplot(data=df, x='lead_time', bins=35, kde=True)
plt.title('Lead Time Distribution')
plt.show()

In [ ]:
sns.boxplot(data=df, x='booking_status', y='average_price')
plt.title('Average Price by Booking Status')
plt.xticks(rotation=15)
plt.show()

In [ ]:
pd.crosstab(df['market_segment_type'], df['booking_status'], normalize='index').plot(kind='bar', stacked=True, figsize=(9, 4), colormap='copper')
plt.title('Booking Status by Market Segment')
plt.ylabel('Proportion')
plt.show()

In [ ]:
sns.heatmap(df[['lead_time', 'average_price', 'special_requests', 'total_guests', 'total_nights', 'cancellation_ratio']].corr(), annot=True, cmap='YlGnBu', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
monthly_cancel = df.groupby('reservation_month')['booking_status'].apply(lambda x: (x == 'Canceled').mean()).reset_index(name='cancel_rate')
sns.lineplot(data=monthly_cancel, x='reservation_month', y='cancel_rate', marker='o')
plt.title('Cancellation Rate by Reservation Month')
plt.show()

In [ ]:
sns.scatterplot(data=df, x='lead_time', y='average_price', hue='booking_status', alpha=0.5)
plt.title('Lead Time vs Average Price')
plt.show()

# Modeling

In [ ]:
numeric_features = ['number_of_adults', 'number_of_children', 'number_of_weekend_nights', 'number_of_week_nights', 'car_parking_space', 'lead_time', 'repeated', 'previous_cancellations', 'previous_non_cancellations', 'average_price', 'special_requests', 'reservation_month', 'reservation_day', 'reservation_weekday', 'total_guests', 'total_nights', 'cancellation_ratio', 'arrival_is_weekend']
categorical_features = ['type_of_meal', 'room_type', 'market_segment_type', 'lead_time_level']
feature_columns = numeric_features + categorical_features
X = df[feature_columns]
y = (df['booking_status'] == 'Canceled').astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train.shape, X_test.shape

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline(steps=[('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features),
    ('cat', Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OneHotEncoder(handle_unknown='ignore'))]), categorical_features)
])
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)
X_train_processed.shape

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=1)
}
results = []
for name, model in models.items():
    model.fit(X_train_processed, y_train)
    y_pred = model.predict(X_test_processed)
    results.append({
        'model': name,
        'accuracy': round(accuracy_score(y_test, y_pred), 4),
        'precision': round(precision_score(y_test, y_pred), 4),
        'recall': round(recall_score(y_test, y_pred), 4),
        'f1_score': round(f1_score(y_test, y_pred), 4)
    })
comparison_df = pd.DataFrame(results).sort_values('f1_score', ascending=False)
comparison_df

## Parameter Tuning

In [ ]:
grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=1),
    param_grid={'n_estimators': [150, 200], 'max_depth': [10, None], 'min_samples_split': [2, 5]},
    scoring='f1',
    cv=3,
    n_jobs=1
)
grid_search.fit(X_train_processed, y_train)
grid_search.best_params_

## Validation and Evaluation

In [ ]:
best_model = grid_search.best_estimator_
final_predictions = best_model.predict(X_test_processed)
final_scores = {
    'accuracy': round(accuracy_score(y_test, final_predictions), 4),
    'precision': round(precision_score(y_test, final_predictions), 4),
    'recall': round(recall_score(y_test, final_predictions), 4),
    'f1_score': round(f1_score(y_test, final_predictions), 4)
}
final_scores

In [ ]:
print(classification_report(y_test, final_predictions, target_names=['Not Canceled', 'Canceled']))

In [ ]:
sns.heatmap(confusion_matrix(y_test, final_predictions), annot=True, fmt='d', cmap='OrRd')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# Deployment

In [ ]:
comparison_df